### 1.1 Alternative Initialization: 
Effective initialization can be critical for solving nonlinear problems, since they can have several local solutions
and numerical difficulties. Below we solve the Rosenbrock example using different initial values for the x variables. We write a loop that varies the initial value from 2.0 to 6.0, solves the problem, and prints the solution for each iteration of the loop.

In [ ]:
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys

if "google.colab" in sys.modules:
    required_solvers = {'ipopt'}

    if importlib.util.find_spec("pyomo") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyomo"])

    apt_packages = []
    if "glpk" in required_solvers and shutil.which("glpsol") is None:
        apt_packages.append("glpk-utils")
    if "cbc" in required_solvers and shutil.which("cbc") is None:
        apt_packages.append("coinor-cbc")

    if apt_packages:
        subprocess.check_call(["apt-get", "update"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", *apt_packages])

    if "ipopt" in required_solvers and shutil.which("ipopt") is None:
        if importlib.util.find_spec("idaes") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "idaes-pse"])
        idaes_bin = Path.home() / ".idaes" / "bin"
        subprocess.check_call(["idaes", "get-extensions"])
        os.environ["PATH"] = f"{idaes_bin}:{os.environ['PATH']}"

    missing_solvers = []
    if "glpk" in required_solvers and shutil.which("glpsol") is None:
        missing_solvers.append("glpk")
    if "cbc" in required_solvers and shutil.which("cbc") is None:
        missing_solvers.append("cbc")
    if "ipopt" in required_solvers and shutil.which("ipopt") is None:
        missing_solvers.append("ipopt")

    if missing_solvers:
        raise RuntimeError(
            "Missing solver executables after Colab setup: "
            + ", ".join(sorted(missing_solvers))
        )


In [1]:
# rosenbrock_script_loop.py: A Pyomo model for the Rosenbrock problem
import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.x = pyo.Var()
model.y = pyo.Var()

def rosenbrock(m):
    return (1.0-m.x)**2 + 100.0*(m.y - m.x**2)**2
model.obj = pyo.Objective(rule=rosenbrock, sense=pyo.minimize)


solver = pyo.SolverFactory('ipopt')

print('x_init, y_init, x_soln, y_soln')
y_init = 5.0
for x_init in range(2, 6):
    model.x = x_init
    model.y = 5.0

    solver.solve(model)

    print("{0:6.2f}  {1:6.2f}  {2:6.2f}  {3:6.2f}".format(x_init, \
            y_init, pyo.value(model.x), pyo.value(model.y)))


x_init, y_init, x_soln, y_soln
  2.00    5.00    1.00    1.00
  3.00    5.00    1.00    1.00
  4.00    5.00    1.00    1.00
  5.00    5.00    1.00    1.00
